# SER Baselines — Colab Runner

Trains and evaluates the four baselines from the milestone PDF (slide 4):

| # | Baseline | Config | Isolates |
|---|---|---|---|
| B1 | Square 3×3 kernel | `baseline_square_kernel.yaml` | kernel shape |
| B2 | Bidirectional LSTM | `baseline_bilstm.yaml` | causality |
| B3 | Last-frame loss | `baseline_last_frame_loss.yaml` | per-frame supervision |
| B4 | Sharan CNN-BiLSTM | `baseline_sharan.yaml` | published SOTA pattern |

Each cell is independent and idempotent. Run the cells top to bottom; if a baseline fails or you stop midway, re-running the relevant Cell 8x will pick up cleanly.

**Output:** the final cell runs `plot_baselines.py`, which compares all four baselines against your existing proposed-family runs (Baseline / +Harmonic / +FreqPos / +Both) and writes plots into `results/plots/`.

In [ ]:
# Cell 1 — clone repo & install deps
import os

REPO_URL = "https://github.com/rushankgoyal/crnn-ser.git"  # update if your fork is elsewhere
REPO_DIR = "/content/crnn-ser"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!pip install -q -r requirements.txt

In [ ]:
# Cell 2 — GPU check
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU — training will run on CPU and be slow.")

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/cs231n-ser"
os.makedirs(f"{DRIVE_ROOT}/data/ravdess", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
print("Drive mounted. DRIVE_ROOT =", DRIVE_ROOT)

In [ ]:
# Cell 4 — Ensure preprocessed RAVDESS exists (auto-downloads if needed)
DATA_ROOT = f"{DRIVE_ROOT}/data/ravdess"
os.makedirs(DATA_ROOT, exist_ok=True)

missing = [s for s in ["train", "val", "test"]
           if not os.path.exists(f"{DATA_ROOT}/{s}.npz")]

if not missing:
    print("All splits found:")
    for s in ["train", "val", "test"]:
        size = os.path.getsize(f"{DATA_ROOT}/{s}.npz") / 1e6
        print(f"  {s}.npz  {size:.1f} MB")
else:
    print(f"Missing splits {missing} — downloading and preprocessing RAVDESS (~5 min)...")
    RAW_DIR = "/content/ravdess_raw"
    ZIP_PATH = "/content/ravdess.zip"
    ZIP_URL = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip"
    os.makedirs(RAW_DIR, exist_ok=True)
    if not os.path.exists(ZIP_PATH):
        !wget -q --show-progress -O {ZIP_PATH} {ZIP_URL}
    !unzip -q {ZIP_PATH} -d {RAW_DIR}
    !python data/preprocess.py --dataset ravdess --raw_dir {RAW_DIR} --out_dir {DATA_ROOT}
    !rm -rf {RAW_DIR} {ZIP_PATH}
    still_missing = [s for s in ["train", "val", "test"]
                     if not os.path.exists(f"{DATA_ROOT}/{s}.npz")]
    if still_missing:
        raise RuntimeError(f"Preprocessing failed — missing: {still_missing}")
    print("Preprocessing complete.")

In [ ]:
# Cell 5 — Patch baseline configs with the runtime DATA_ROOT
import yaml

BASELINES = [
    ("baseline_square_kernel.yaml",    "baseline_square_kernel"),
    ("baseline_bilstm.yaml",           "baseline_bilstm"),
    ("baseline_last_frame_loss.yaml",  "baseline_last_frame_loss"),
    ("baseline_sharan.yaml",           "baseline_sharan"),
]
RUNTIME = {}
os.makedirs("/content/runtime_configs", exist_ok=True)

for cfg_file, run_name in BASELINES:
    with open(f"configs/{cfg_file}") as f:
        cfg = yaml.safe_load(f)
    cfg["data_root"] = DATA_ROOT
    cfg["run_name"] = run_name
    out = f"/content/runtime_configs/{run_name}.yaml"
    with open(out, "w") as f:
        yaml.dump(cfg, f)
    RUNTIME[run_name] = out
    print(f"  {run_name:<28}  data_root={DATA_ROOT}")

In [ ]:
# Cell 6 — CPU unit tests (verifies new baseline knobs + Sharan model)
!python tests/test_components.py

In [ ]:
# Cell 7 — Smoke train: 2 epochs of the Sharan baseline to confirm GPU path works
import subprocess, sys
smoke_cfg_path = RUNTIME["baseline_sharan"]
with open(smoke_cfg_path) as f:
    scfg = yaml.safe_load(f)
scfg["train"]["epochs"] = 2
scfg["run_name"] = "smoke_sharan"
smoke_out = "/content/runtime_configs/smoke.yaml"
with open(smoke_out, "w") as f:
    yaml.dump(scfg, f)
res = subprocess.run([sys.executable, "train.py", "--config", smoke_out])
if res.returncode != 0:
    raise RuntimeError(f"Smoke train FAILED (exit {res.returncode})")
print("\nSmoke OK — GPU path confirmed.")

## Train each baseline

Each cell trains one baseline. You can run them in any order or skip any of them; the plot script later picks up only the runs that have a `metrics.json`.

In [ ]:
# Cell 8a — B1: Square 3×3 kernel
!python train.py --config {RUNTIME['baseline_square_kernel']}

In [ ]:
# Cell 8b — B2: Bidirectional LSTM
!python train.py --config {RUNTIME['baseline_bilstm']}

In [ ]:
# Cell 8c — B3: Last-frame loss
!python train.py --config {RUNTIME['baseline_last_frame_loss']}

In [ ]:
# Cell 8d — B4: Sharan CNN-BiLSTM
!python train.py --config {RUNTIME['baseline_sharan']}

In [ ]:
# Cell 9 — Save baseline checkpoints to Drive (so re-evaluation is free next session)
import shutil
for run_name in RUNTIME:
    src = f"runs/{run_name}"
    dst = f"{DRIVE_ROOT}/runs/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved runs/{run_name} → Drive")
    else:
        print(f"  Skipped {run_name} (not trained yet)")

In [ ]:
# Cell 10 — Evaluate every baseline that has a checkpoint
for run_name, cfg_path in RUNTIME.items():
    ckpt = f"runs/{run_name}/best.pt"
    if not os.path.exists(ckpt):
        print(f"  SKIP {run_name}: no checkpoint")
        continue
    print(f"\n{'='*60}\nEvaluating: {run_name}\n{'='*60}")
    !python evaluate.py --config {cfg_path} --checkpoint {ckpt}

In [ ]:
# Cell 11 — Pull proposed-family results from Drive (so plots show both groups)
# If you already ran the ablation notebook, those results are saved to Drive.
# Copy them locally so plot_baselines.py picks them up.
PROPOSED_RUNS = ["ravdess_baseline", "ravdess_harmonic", "ravdess_freqpos", "ravdess_harmonic_freqpos"]
for run_name in PROPOSED_RUNS:
    src = f"{DRIVE_ROOT}/results/{run_name}"
    dst = f"results/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Pulled {run_name} from Drive")
    else:
        print(f"  Missing on Drive: {src} (plot will skip {run_name})")

In [ ]:
# Cell 12 — Generate baseline-vs-proposed plots + summary table
!python plot_baselines.py --results_dir results --out_dir results/plots

from IPython.display import Image, display
for fname in [
    "baselines_latency_accuracy.png",
    "baselines_uar_bar.png",
    "baselines_confusion_grid.png",
    "baselines_first_correct.png",
]:
    p = f"results/plots/{fname}"
    if os.path.exists(p):
        print("\n" + fname)
        display(Image(p))

In [ ]:
# Cell 13 — Sync results + plots to Drive
for run_name in RUNTIME:
    src = f"results/{run_name}"
    dst = f"{DRIVE_ROOT}/results/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved results/{run_name} → Drive")

src_plots = "results/plots"
dst_plots = f"{DRIVE_ROOT}/results/plots"
if os.path.exists(src_plots):
    shutil.copytree(src_plots, dst_plots, dirs_exist_ok=True)
    print(f"  Saved results/plots → Drive")